# 02. $p$-variation

**Scope.** Computing the roughness of a path from samples, and what the answer means.

Separate from notebook 01, which estimates $\|f\|_{L^p}$, a measure of size. $p$-variation measures roughness, is not an integral, and its computation is a live algorithmic question rather than textbook quadrature.

Relevance to the project: the $p$-variation index fixes how many signature levels are needed (§1), floors the sampling budget (`docs/logbook/2026-08-12.md`), and decides whether the rough-path objection to pointwise losses applies to our data at all. It is also the part of the project unaffected by the uniform-sampling assumption of 13/08, since $V_p$ ignores sample times.

**Contents.** §1 definition and standard facts. §§2 to 4 the three implementations in `src/pathloss/pvar.py`, in order of increasing speed and decreasing obviousness. §5 the one-dimensional algorithm we have not implemented. §6 timings. §7 the estimator none of them provide.

---

In [ ]:
%matplotlib inline
import sys, pathlib, time
import numpy as np
import matplotlib.pyplot as plt

# `pathloss` comes from `pip install -e .`; this line also makes the notebook
# work without it.
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from pathloss.pvar import p_variation_brute, p_variation_exact, p_variation_pruned
from pathloss.paths import brownian_motion

rng = np.random.default_rng(0)
plt.rcParams.update({"figure.figsize": (10, 3.6), "figure.dpi": 110})

## 1. Definition and standard facts

For a sampled path $x_0, \dots, x_N$ in a metric space $(E, d)$ and $p \ge 1$,

$$V_p(x)^p \;=\; \sup \sum_k d\big(x_{n_k}, x_{n_{k-1}}\big)^p ,$$

the supremum over increasing subsequences $0 = n_0 < n_1 < \cdots < n_K = N$.

### 1.1 The supremum, and the difficulty at $p>1$

A sum over the given grid is a property of the grid. The supremum is a property of the path: relabelling $t$ permutes which partitions are available without changing the set of achievable sums, so $V_p$ is invariant under reparametrisation.

The exponent decides whether the supremum is trivial. For increments $a, b$ of the same sign,

$$|a+b|^p \;\begin{cases} = |a|^p + |b|^p, & p = 1 \\ > |a|^p + |b|^p, & p > 1 .\end{cases}$$

At $p = 1$ refining never loses, the finest partition wins, and $V_1$ is a cumulative sum: linear in $N$. At $p > 1$ coarsening can gain, so the optimum is some intermediate partition and finding it is an optimisation. Butkus & Norvaiša (2018, p. 361) give the smallest example: a four-value path at $p=3$ where the partition using *every* local extremum scores $17/27$ while the endpoints alone score $1$.

Ferrucci, Perrée & Lyons (2026, Remark 2.2) state the same divide: for $p=1$ an optimal split is a cumulative-sum problem and hence linear, whereas *"for the $p$-variation control with $p>1$ the analogous problem is computationally harder, since even computing the ordinary $p$-variation requires an optimisation over partitions."*

### 1.2 Standard facts

- $V_p$ is non-increasing in $p$, opposite monotonicity to $\|f\|_{L^p}$.
- $p = 1$ is total variation. $V_1 < \infty$ iff Riemann-Stieltjes integration against $f$ is defined.
- Brownian motion has $V_p < \infty$ a.s. exactly for $p > 2$, so $p = 2$ is critical.
- $\alpha$-Hölder $\Rightarrow V_{1/\alpha} < \infty$, since $\alpha p = 1$ makes $\sum_i (\Delta t_i)^{\alpha p}$ telescope to $T$ for every partition. The converse fails without reparametrisation (`docs/logbook/2026-08-12.md`, Reading 2).
- A path of finite $p$-variation needs $\lfloor p \rfloor$ signature levels and no more; higher levels are determined (Lyons, 1998). So the index says how much of the signature is data.

### 1.3 Shared by every implementation below

All three take the supremum over subsequences of the **observed grid**. The true $V_p$ allows any partition of $[0,T]$, including points between samples, so every number here is a **lower bound**, and no finite sample determines the truth. Same limitation as notebook 01 §3.1, arriving by a different route.

---

## 2. Implementation 1: enumeration

`p_variation_brute(x, p, dist=None, max_points=20)`

### Definition

Lists every subsequence containing both endpoints, scores each, returns the largest. With $N+1$ points the interior has $N-1$ optional members, so $2^{N-1}$ candidates.

### Purpose

Nothing in it can be subtly wrong: it contains no argument, only a loop over every candidate. It is the oracle the other two are tested against in `tests/test_pvar.py`, and it is the reason those tests mean anything. Before it existed, every $p$-variation test compared one of my implementations against another, or against a monotone path, where the winning candidate is the first one tried and a bug in the search would not show.

### Cost, and the guard

$2^{N-1}$ scored subsequences, each costing $O(N)$. Refuses inputs above 20 points by default, deliberately: a test slow enough to be skipped is a test that stops being run.

### Reading the code

```python
for mask in range(1 << interior):        # every subset of the interior
    idx = [0] + [i + 1 for i in range(interior) if (mask >> i) & 1] + [n - 1]
    total = sum(d(a, b) ** p for a, b in zip(idx[:-1], idx[1:]))
```

The bit mask is the subset. Endpoints are prepended and appended rather than being optional, since the definition fixes $n_0 = 0$ and $n_K = N$.

---

## 3. Implementation 2: dynamic programming

`p_variation_exact(x, p, dist=None, return_points=False)`

### The observation that removes the exponential

Stop asking *which subset*, and ask instead, for each index $j$, what the best score is for a subsequence **ending at** $j$. Call it $D[j]$.

Any such subsequence has a last step, from some $m < j$. Fix $m$. Then everything before $m$ is itself a subsequence ending at $m$, and it must be optimal: substituting a better prefix increases the total and leaves the last step $d(x_m, x_j)^p$ untouched. So

$$D[0] = 0, \qquad D[j] \;=\; \max_{m<j}\big(D[m] + d(x_m,x_j)^p\big), \qquad V_p = D[N]^{1/p} .$$

This is optimal substructure, and it is what makes the problem a dynamic programme rather than a search. The exponential disappears because the choice of prefix and the choice of last step do not interact except through $m$.

### Cost

$N$ values of $j$, each scanning up to $N$ candidates: $O(N^2)$ time, $O(N)$ memory beyond the path.

### Reading the code

```python
for j in range(1, n):
    incr = np.linalg.norm(x[j] - x[:j], axis=-1) ** p   # d(x_m, x_j)^p for all m < j
    cand = best[:j] + incr                              # D[m] + that
    k = int(np.argmax(cand))
    best[j], link[j] = cand[k], k
```

One vectorised row per $j$. `link` records the arg-max so the maximising subsequence can be recovered by walking backwards from $N$; `return_points=True` does that. The links are bookkeeping carried alongside the value and can drift from it, which is why a test checks that the reported subsequence achieves the reported number.

### Two properties visible from the recursion

**Monotonicity.** $D[j+1] \ge D[j] + d(x_j,x_{j+1})^p \ge D[j]$, so $D$ is non-decreasing. §4 is built on this.

**A supplied metric costs.** With `dist=None` the row is one NumPy call. With a metric supplied it becomes $j$ Python calls, so the constant rises by orders of magnitude while the asymptotics hold.

---

## 4. Implementation 3: pruned search

`p_variation_pruned(x, p, dist=None, return_points=False)`

A port of `p_var_backbone` from [khumarahn/p-var](https://github.com/khumarahn/p-var) (Korepanov, Lyons, Zorin-Kranich). Same recursion as §3, computed without visiting most of the candidates.

### 4.1 The bar, and its rise

Scan $m$ downward from $j-1$, keeping $\texttt{best}$, the largest value found so far for endpoint $j$. Candidate $m$ improves on it only if

$$D[m] + d(x_m,x_j)^p > \texttt{best}
\qquad\Longleftrightarrow\qquad
d(x_m,x_j) \;>\; \big(\texttt{best} - D[m]\big)^{1/p} \;=:\; \delta .$$

$D$ is non-decreasing (§3), so $D[m]$ falls as $m$ falls, so $\delta$ **rises**. An older index arrives carrying less banked score, so its one hop has to make up the difference. The further back the scan goes, the more a candidate must be worth to be worth anything.

### 4.2 Excluding a block with one distance evaluation

Partition the indices into dyadic blocks. For a block $B$ with a chosen centre $c$ and radius

$$R_B = \max_{m' \in B} d(x_{m'}, x_c),$$

the triangle inequality gives, for every $m' \in B$,

$$d(x_{m'}, x_j) \;\le\; d(x_{m'}, x_c) + d(x_c, x_j) \;\le\; R_B + d(x_c, x_j).$$

If that upper bound is at most $\delta$, no member of $B$ clears the bar and all $|B|$ of them are skipped on one distance evaluation. Searching for a distant point by postcode rather than by door.

Only the triangle inequality and symmetry are used. That is why `dist` may be any metric, and it is the property the project will need: rough-path $p$-variation is measured in the homogeneous norm on signatures rather than on values, and the same routine computes it.

### 4.3 The role of dyadic blocks

Each index lies in exactly $\log N$ dyadic blocks, and blocks at different levels nest. So the radii can be accumulated online as $j$ advances, at $\log N$ updates per step, and the scan can try the largest block first and shrink until the bound bites. `radius` packs all levels into one flat array of length $N-1$ at position `(s >> k) + (j >> k)`, with a guard for the truncated final block.

Note "dyadic" here is the shape of a search tree. It changes no answer, unlike a restriction on which partitions are considered.

### 4.4 Cost, and the worst case

About $O(N\log N)$ on a path that changes direction. $O(N^2)$ on a monotone path in $\mathbb{R}$, where

$$d(x_{m'}, x_c) + d(x_c, x_j) = d(x_{m'}, x_j)$$

**exactly**: the bound has no slack, so nothing is ever excluded. Pruning needs the triangle inequality to be strict, and on a straight line it is an equality. Conversely a wandering path returns near where it has been, so going further back does not take you much further away while the bar keeps rising, and whole blocks fall away.

### 4.5 One implementation detail worth knowing

$\delta$ is refreshed lazily. A $\delta$ computed at a later $m$ is smaller than the true current one, hence conservative, so the cheap comparison runs first and the $p$-th root is taken only when it fails. Correctness survives because using too small a $\delta$ can only admit candidates that are then rejected by the explicit comparison.

---

## 5. Not implemented: the one-dimensional algorithm

Butkus & Norvaiša (2018), R package `pvar`. A third strategy: rather than searching more cleverly, **make the problem smaller**.

**Step 1, corners.** A maximising partition is always a subpartition of the minimal monotonicity partition (their Theorem 1), so every point that is not a local extremum can be deleted outright. One linear pass, and for smooth data it removes most of the input.

**Step 2, redundant corners.** Even some turning points are droppable. Over a window $S_k, \dots, S_{k+m}$, if

$$\sum_{i=k+1}^{k+m} |S_i - S_{i-1}|^p \;<\; |S_{k+m} - S_k|^p ,$$

then hopping straight across beats the interior, so some inner point is redundant (their Corollary 4), and their Lemmas 4 and 5 upgrade "some" to "all of them". Sweep $m = 3, 5, 7, \dots$, deleting, then merge the surviving pieces pairwise.

**Obstruction for us.** Local extrema, monotone runs and alternating signs all require an ordered line. A path in $\mathbb{R}^d$ has no local maxima. This is what the p-var README means by "it does not work for example in $\mathbb{R}^2$. Here we rectify this."

**Value to us regardless.** Its best case is §4's worst case. Smooth or monotone data is annihilated by step 1 and defeats the block bound of §4.2. If real data turns out to be smooth (13/08 makes univariate real series the starting point), the R package is the faster route and `FindCorners` is a ten-line preprocessing step we could adopt for $d=1$ regardless.

---

## 6. Cost, measured

§3 and §4 return the same number, so the comparison below is about time alone. The assertion in the cell states that.

Read the growth rate rather than the absolute times: $O(N^2)$ against roughly $O(N\log N)$, so the ratio should widen with $N$.

**Caveat.** `p_variation_exact` is one NumPy call per row; `p_variation_pruned` is a Python loop. The constant favours the first heavily, so the crossover sits at much larger $N$ than the asymptotics alone suggest, and these timings measure the implementations rather than the algorithms.

In [ ]:
sizes = [2**k + 1 for k in range(7, 12)]
p = 2.5
_, W = brownian_motion(n=sizes[-1], T=1.0, d=1, rng=3)

print(f"{'n':>7} {'V_p':>10} {'t_exact':>10} {'t_pruned':>10} {'ratio':>8}")
for n in sizes:
    w = W[:n]
    t0 = time.perf_counter(); ve = p_variation_exact(w, p); t1 = time.perf_counter()
    vp = p_variation_pruned(w, p);                          t2 = time.perf_counter()
    assert abs(ve - vp) < 1e-9 * max(1.0, ve), "implementations disagree"
    print(f"{n:>7} {ve:>10.4f} {t1-t0:>10.4f} {t2-t1:>10.4f} {(t1-t0)/(t2-t1):>8.2f}")

## 7. Not implemented: estimating the index

Every function above takes $p$ as an argument. What the project needs is

$$p^\ast = \inf\{p : V_p < \infty\},$$

which no single evaluation supplies: $V_p$ computed on $N$ points is finite for every $p$. The index shows in how the sum **diverges under refinement**.

### The estimator

If increments over windows of width $2^{-k}$ scale as $2^{-k/p^\ast}$, then summing $2^k$ of them,

$$\sum_{i=1}^{2^k} |\Delta_i x|^p \;\approx\; 2^{k}\cdot 2^{-kp/p^\ast} \;=\; 2^{\,k(1 - p/p^\ast)} .$$

The exponent is positive for $p < p^\ast$, negative for $p > p^\ast$, zero at $p = p^\ast$. So regress $\log_2$ of the level-$k$ sum on $k$; the slope is $1 - p/p^\ast$; the $p$ at which it crosses zero is the index.

This is the same trichotomy as §1.2's Hölder computation, in dyadic notation. There, $\sum(\Delta t)^{q} = T h^{q-1}$ blows up for $q<1$, is constant at $q=1$ and vanishes for $q>1$. The estimator measures which of the three cases holds at each $p$ and reports the crossover.

### Two known failure modes

**Observation noise.** Additive noise has infinite variation, so at fine scales it dominates and drives the estimate upward. This is the microstructure problem from realised-volatility estimation. Consequence: the answer depends on the range of scales fitted, and that range must be reported alongside it.

**Downward bias.** The supremum over the observed grid understates the true supremum, so every $V_p$ here is a lower bound and the index inherits the bias.

### Priority

$p^\ast$ floors the sampling budget at $N \gtrsim \epsilon^{-p^\ast}$ (`docs/logbook/2026-08-12.md`, a floor rather than a rate, since finite $p$-variation bounds the Hölder exponent above without fixing it), sets the signature truncation level at $\lfloor p^\ast \rfloor$, and decides whether the rough-path objection to pointwise losses applies to our data. It is also the first quantity in the project that is about the data rather than about our own constructions.

---

## References

Keyed to `papers/references.bib`.

- ★ **Ferrucci, Perrée & Lyons (2026)**, arXiv:2607.26281, Remark 2.2: why $V_p$ for $p>1$ is an optimisation over partitions. §1.1.
- **Korepanov, Lyons & Zorin-Kranich**, [p-var](https://github.com/khumarahn/p-var): the pruned algorithm, valid in any metric space. §4.
- **Butkus & Norvaiša (2018)**, *Computation of $p$-variation*, Lith. Math. J. 58(4), with R package `pvar`. §1.1, §5.
- **Daoudi & Junca (2024)**, *Efficient algorithms computing $p$-variation*, preprint. §5.
- **Lyons (1998)**, *Differential equations driven by rough signals*, Rev. Mat. Iberoamericana 14(2): $\lfloor p \rfloor$ levels suffice. §1.2.